# 1. Prepare the paired MEA dataset

90 paired wells; 44 MEA features. Outputs: Mean + Delta (88), Delta-only (44), and MFR (2).

## Setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "mea_ml" / "io.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

from typing import List, Tuple

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from mea_ml.io import ROOT as PROJECT_ROOT
from mea_ml.training import RANDOM_STATE
INPUT_PATH = PROJECT_ROOT / "data" / "paired_mea_features_raw.csv"
OUT_DIR = PROJECT_ROOT / "results" / "01_dataset"
OUT_DIR.mkdir(parents=True, exist_ok=True)

from mea_ml.features import (FEATURE_ORDER, FEATURE_FAMILY_MAP, PARENT_METRIC_MAP, MFR_FEATURES, META_COLUMNS)

print("Input:", INPUT_PATH.relative_to(PROJECT_ROOT))
print("Output directory:", OUT_DIR.relative_to(PROJECT_ROOT))


Input: data/paired_mea_features_raw.csv
Output directory: results/01_dataset


## Validate input

In [2]:
paired_raw = pd.read_csv(INPUT_PATH)

required_feature_columns = [
    f"{state}::{feature}"
    for state in ["Base", "Treat"]
    for feature in FEATURE_ORDER
]
required_columns = META_COLUMNS + required_feature_columns
missing_columns = sorted(set(required_columns) - set(paired_raw.columns))
if missing_columns:
    raise ValueError(f"Input is missing required columns: {missing_columns}")

if paired_raw.duplicated(["group_id", "well"]).any():
    raise ValueError("Each group/well combination must occur exactly once.")
if paired_raw.groupby("group_id")["class"].nunique().max() != 1:
    raise ValueError("Each recording group must have one class label.")

expected_wells = {"A1", "A2", "A3", "B1", "B2", "B3"}
class_counts = paired_raw["class"].value_counts().to_dict()
group_sizes = paired_raw.groupby("group_id").size()
group_wells = paired_raw.groupby("group_id")["well"].agg(lambda values: set(values.astype(str)))
group_class_counts = paired_raw[["group_id", "class"]].drop_duplicates()["class"].value_counts().to_dict()
if class_counts != {"Negative": 30, "Positive": 60}:
    raise ValueError(f"Expected 30 Negative and 60 Positive organoids; got {class_counts}.")
if group_class_counts != {"Negative": 5, "Positive": 10}:
    raise ValueError(f"Expected 5 Negative and 10 Positive recording groups; got {group_class_counts}.")
if not group_sizes.eq(6).all() or not group_wells.map(lambda wells: wells == expected_wells).all():
    raise ValueError("Every recording group must contain wells A1–A3 and B1–B3 exactly once.")

raw_feature_data = paired_raw[required_feature_columns].copy()
numeric_feature_data = raw_feature_data.apply(pd.to_numeric, errors="coerce")
invalid_nonblank = raw_feature_data.notna() & numeric_feature_data.isna()
if invalid_nonblank.any().any():
    row_idx, col_idx = np.argwhere(invalid_nonblank.to_numpy())[0]
    raise ValueError(
        f"Non-numeric feature value at row {paired_raw.index[row_idx]}, "
        f"column {required_feature_columns[col_idx]}."
    )
nonfinite = numeric_feature_data.notna() & ~np.isfinite(numeric_feature_data)
if nonfinite.any().any():
    raise ValueError("Feature values must be finite or blank.")
paired_raw[required_feature_columns] = numeric_feature_data

print("Paired organoids:", len(paired_raw))
print("Recording groups:", paired_raw["group_id"].nunique())
print("Class counts:")
display(paired_raw["class"].value_counts().rename_axis("class").to_frame("n"))
print("Structurally undefined AxIS values before zero-fill:", int(paired_raw[required_feature_columns].isna().sum().sum()))
display(paired_raw[META_COLUMNS].head())


Paired organoids: 90
Recording groups: 15
Class counts:


,n
class,
Positive,60
Negative,30


Structurally undefined AxIS values before zero-fill: 2574


,class,pair_stem,well,group_id,sample_id_base,sample_id_treat
0,Positive,POS01,A1,POS01,POS01|Base|A1,POS01|Treat|A1
1,Positive,POS01,A2,POS01,POS01|Base|A2,POS01|Treat|A2
2,Positive,POS01,A3,POS01,POS01|Base|A3,POS01|Treat|A3
3,Positive,POS01,B1,POS01,POS01|Base|B1,POS01|Treat|B1
4,Positive,POS01,B2,POS01,POS01|Base|B2,POS01|Treat|B2


## Zero-fill undefined AxIS values

In [3]:
paired_filled = paired_raw.copy()
manifest_rows = []

for feature in FEATURE_ORDER:
    base_column = f"Base::{feature}"
    treat_column = f"Treat::{feature}"
    n_missing_base = int(paired_filled[base_column].isna().sum())
    n_missing_treat = int(paired_filled[treat_column].isna().sum())
    paired_filled[[base_column, treat_column]] = paired_filled[[base_column, treat_column]].fillna(0.0)
    manifest_rows.append({
        "raw_feature": feature,
        "feature_family": FEATURE_FAMILY_MAP[feature],
        "parent_metric": PARENT_METRIC_MAP[feature],
        "n_missing_base_raw": n_missing_base,
        "n_missing_treat_raw": n_missing_treat,
        "zero_fill_rule": "fill_missing_with_0",
        "retained_for_primary_analysis": True,
    })

feature_manifest = pd.DataFrame(manifest_rows)
feature_catalog = feature_manifest[["raw_feature", "feature_family", "parent_metric"]].copy()
feature_catalog["preprocessing_rule"] = "AxIS structurally undefined values filled with 0"

if paired_filled[required_feature_columns].isna().any().any():
    raise RuntimeError("Missing values remain after zero-fill.")

display(feature_manifest.head())
print("Retained MEA features:", len(feature_manifest))


,raw_feature,feature_family,parent_metric,n_missing_base_raw,n_missing_treat_raw,zero_fill_rule,retained_for_primary_analysis
0,Burst Duration - Avg (s),burst,Number of Bursts,8,26,fill_missing_with_0,True
1,Burst Duration - Std (s),burst,Number of Bursts,14,27,fill_missing_with_0,True
2,Burst Frequency - Avg (Hz),burst,Number of Bursts,8,26,fill_missing_with_0,True
3,Burst Frequency - Std (Hz),burst,Number of Bursts,14,27,fill_missing_with_0,True
4,Burst Percentage - Avg,burst,Number of Bursts,0,4,fill_missing_with_0,True


Retained MEA features: 44


## Construct feature matrices

In [4]:
def build_engineered_matrix(
    paired_df: pd.DataFrame,
    raw_features: List[str],
    feature_mode: str,
) -> Tuple[pd.DataFrame, List[str]]:
    base = paired_df[[f"Base::{feature}" for feature in raw_features]].to_numpy(dtype=float)
    treat = paired_df[[f"Treat::{feature}" for feature in raw_features]].to_numpy(dtype=float)
    mean = 0.5 * (base + treat)
    delta = treat - base

    if feature_mode == "mean+delta":
        values = np.hstack([mean, delta])
        names = [f"Mean::{feature}" for feature in raw_features] + [f"Delta::{feature}" for feature in raw_features]
    elif feature_mode == "delta-only":
        values = delta
        names = [f"Delta::{feature}" for feature in raw_features]
    elif feature_mode == "mfr-only":
        indices = [raw_features.index(feature) for feature in MFR_FEATURES]
        values = np.hstack([mean[:, indices], delta[:, indices]])
        names = [f"Mean::{feature}" for feature in MFR_FEATURES] + [f"Delta::{feature}" for feature in MFR_FEATURES]
    else:
        raise ValueError(f"Unknown feature mode: {feature_mode}")

    return pd.DataFrame(values, columns=names), names

retained_features = feature_manifest["raw_feature"].tolist()
X_mean_delta_df, feature_names_mean_delta = build_engineered_matrix(paired_filled, retained_features, "mean+delta")
X_delta_only_df, feature_names_delta_only = build_engineered_matrix(paired_filled, retained_features, "delta-only")
X_mfr_only_df, feature_names_mfr_only = build_engineered_matrix(paired_filled, retained_features, "mfr-only")

y = paired_filled["class"].eq("Positive").astype(int).to_numpy()
groups = paired_filled["group_id"].astype(str).to_numpy()

print("Mean + Delta:", X_mean_delta_df.shape)
print("Delta-only:", X_delta_only_df.shape)
print("MFR comparator:", X_mfr_only_df.shape)


Mean + Delta: (90, 88)
Delta-only: (90, 44)
MFR comparator: (90, 2)


## Save dataset

In [5]:
group_manifest = (
    paired_filled[["class", "pair_stem", "group_id"]]
    .drop_duplicates()
    .sort_values(["class", "group_id"])
    .reset_index(drop=True)
)
group_manifest["n_paired_wells"] = group_manifest["group_id"].map(paired_filled.groupby("group_id").size())
pair_manifest = paired_filled[META_COLUMNS].copy()

def training_table(matrix: pd.DataFrame) -> pd.DataFrame:
    return pd.concat(
        [paired_filled[META_COLUMNS].reset_index(drop=True), pd.Series(y, name="y"), matrix.reset_index(drop=True)],
        axis=1,
    )

paired_filled.to_csv(OUT_DIR / "paired_mea_features_zero_filled.csv", index=False)
training_table(X_mean_delta_df).to_csv(OUT_DIR / "training_matrix_mean_delta.csv", index=False)
training_table(X_delta_only_df).to_csv(OUT_DIR / "training_matrix_delta_only.csv", index=False)
training_table(X_mfr_only_df).to_csv(OUT_DIR / "training_matrix_mfr.csv", index=False)
feature_manifest.to_csv(OUT_DIR / "feature_manifest.csv", index=False)
feature_catalog.to_csv(OUT_DIR / "feature_catalog.csv", index=False)
group_manifest.to_csv(OUT_DIR / "group_manifest.csv", index=False)
pair_manifest.to_csv(OUT_DIR / "pair_manifest.csv", index=False)

bundle = {
    "X_mean_delta_df": X_mean_delta_df.copy(),
    "X_delta_only_df": X_delta_only_df.copy(),
    "X_mfr_only_df": X_mfr_only_df.copy(),
    "X_mean_delta": X_mean_delta_df.to_numpy(dtype=float),
    "X_delta_only": X_delta_only_df.to_numpy(dtype=float),
    "X_mfr_only": X_mfr_only_df.to_numpy(dtype=float),
    "feature_names_mean_delta": feature_names_mean_delta,
    "feature_names_delta_only": feature_names_delta_only,
    "feature_names_mfr_only": feature_names_mfr_only,
    "y": y.copy(),
    "groups": groups.copy(),
    "paired_meta": paired_filled[META_COLUMNS].copy(),
    "retained_raw_features": retained_features.copy(),
    "feature_manifest": feature_manifest.copy(),
    "feature_catalog": feature_catalog.copy(),
    "group_manifest": group_manifest.copy(),
    "pair_manifest": pair_manifest.copy(),
    "primary_feature_set": "mean+delta",
    "preprocessing_policy": "AxIS structurally undefined outputs deterministically filled with 0",
    "random_state": RANDOM_STATE,
}
joblib.dump(bundle, OUT_DIR / "training_bundle.joblib")

print("Wrote", len(list(OUT_DIR.iterdir())), "files to", OUT_DIR.relative_to(PROJECT_ROOT))


Wrote 9 files to results/01_dataset
